# Practice Assignment - Understanding torch.compile

### Experiments
4. First-call compilation cost versus warmed execution.
5. Write a fusion-friendly workload and test whether compilation helps.
6. Deliberately break graph capture and inspect the result.

An optional extension explores guards and recompilation if you finish early.


## Before you start

Use **Runtime > Change runtime type > GPU**.

For every experiment:

- Predict before running the measurement.
- Change only the variable named in the task.
- Use repeated measurements rather than one request.
- Compare results from your own Colab runtime. Different students may receive different GPUs.

If a TODO is difficult, open Hint 1 first. Use Hint 2 only if you still need help.


## Setup

The next cells are **plumbing code**. Run them as-is.

In [ ]:
import statistics
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch._dynamo as dynamo
from torch import profiler

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime and run the notebook again")

print("PyTorch", torch.__version__)
print("GPU", torch.cuda.get_device_name(0))
print("CUDA", torch.version.cuda)

RESULTS_DIR = Path("/content/torch_compile_lab_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def benchmark_ms(fn, *args, warmup=8, iters=30):
    with torch.inference_mode():
        for _ in range(warmup):
            fn(*args)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(iters):
            fn(*args)
        torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - start) / iters


def timed_first_call_ms(fn, *args):
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        fn(*args)
    torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - start)


def profile_table(fn, *args, warmup=5, rows=12):
    with torch.inference_mode():
        for _ in range(warmup):
            fn(*args)
        torch.cuda.synchronize()
        with profiler.profile(
            activities=[profiler.ProfilerActivity.CPU, profiler.ProfilerActivity.CUDA],
            record_shapes=True,
        ) as p:
            for _ in range(3):
                fn(*args)
            torch.cuda.synchronize()
    print(p.key_averages().table(sort_by="self_cuda_time_total", row_limit=rows))
    return p

# Experiment 4 - Is compiled execution always faster?

**Concept reacp:** PyTorch eager mode executes operations dynamically. `torch.compile` captures compilable regions and lets a backend optimize them. Compilation itself takes time, so the first compiled call and warmed steady-state calls answer different questions.

### Your Prediction
Which will be slower: the first compiled call or a warmed compiled call? Can a function have better steady-state performance and still be a bad choice to compile if it runs only once?


> **Your response**
>
> Write your response here


In [ ]:
torch.manual_seed(0)
x = torch.randn(4096, 4096, device="cuda")
y = torch.randn(4096, 4096, device="cuda")


def provided_workload(a, b):
    p = a * b
    q = p + a
    r = torch.sin(q)
    return torch.relu(r) * 0.5 + 1.0

### Task 4

Complete the benchmark. You must create a compiled version of `provided_workload`, measure its first invocation separately, and then measure warmed eager and compiled execution.


In [ ]:
torch._dynamo.reset()

# TODO 1: Compile provided_workload.
compiled_workload = None

# TODO 2: Measure the first call of the compiled function.
first_compiled_ms = None

# TODO 3: Measure steady-state eager and compiled execution.
eager_steady_ms = None
compiled_steady_ms = None

# TODO 4: Put the three measurements into a dataframe called compile_cost_df.
compile_cost_df = pd.DataFrame()

display(compile_cost_df)

<details>
<summary>Hint 1</summary>

Use `torch.compile` once. Use `timed_first_call_ms` only before the compiled function has been warmed. Then use `benchmark_ms` for both steady-state measurements.

</details>


<details>
<summary>Hint 2</summary>

Pseudocode:

    compiled function = compile original function
    first time = timed first call compiled function
    eager steady = benchmark original function
    compiled steady = benchmark compiled function
    create three row dataframe

</details>


In [ ]:
required = {"mode", "ms"}
if isinstance(compiled_workload, object) and compiled_workload is not None:
    if (
        isinstance(compile_cost_df, pd.DataFrame)
        and len(compile_cost_df) == 3
        and required.issubset(compile_cost_df.columns)
    ):
        print("PASS")
    else:
        print("CHECK AGAIN")
else:
    print("CHECK AGAIN")

### Explain Experiment 4

1. How much larger was the first compiled call than the warmed compiled call?
2. Calculate warmed speedup as eager time divided by compiled time. A value below 1 means a slowdown.
3. Why is the first compiled call not a fair measure of steady-state performance?
4. When could eager execution still give a lower total time-to-result?


> **Your response**
>
> Write your response here


# Experiment 5 - Write something the compiler can optimize

**Concept reacp:** TorchInductor can optimize across a captured graph. For chains of elementwise operations, one opportunity is operator fusion: doing several operations in fewer generated kernels can reduce launch overhead and intermediate memory traffic.

### Your Prediction
What kind of PyTorch function would give fusion a clearer opportunity than a single matrix multiplication?


> **Your response**
>
> Write your response here


### Task 5A - Write your own fusion-friendly function

Write `my_fusion_fn(a, b)` with at least five tensor operations. Keep it elementwise: examples of possible building blocks include arithmetic, `sin`, `cos`, `tanh`, `sigmoid`, and `relu`.

Do not copy `provided_workload` exactly. Make your own chain.


In [ ]:
def my_fusion_fn(a, b):
    # TODO: Write your own chain of at least five elementwise tensor operations.
    # The function must return a tensor with the same shape as a and b.
    return None

<details>
<summary>Hint 1</summary>

A useful pattern is to create one intermediate tensor, apply several unary or binary elementwise operations, and return a final tensor. Avoid Python loops over tensor elements.

</details>


<details>
<summary>Hint 2</summary>

Pseudocode only:

    z = elementwise operation on a and b
    z = another operation on z
    z = another operation using a or b
    z = nonlinear operation
    z = final elementwise operation
    return z

</details>


In [ ]:
try:
    test_out = my_fusion_fn(x, y)
    if isinstance(test_out, torch.Tensor) and test_out.shape == x.shape:
        print("PASS")
    else:
        print("CHECK AGAIN")
except Exception:
    print("CHECK AGAIN")

### Task 5B - Benchmark and profile your function

Compile your function, benchmark eager and compiled steady-state execution, and inspect both profiler tables. Store the two benchmark rows in `fusion_df`.


In [ ]:
torch._dynamo.reset()

# TODO 1: Compile my_fusion_fn.
compiled_fusion_fn = None

# TODO 2: Benchmark eager and compiled steady-state execution.
my_eager_ms = None
my_compiled_ms = None

# TODO 3: Create a two-row dataframe called fusion_df.
fusion_df = pd.DataFrame()

display(fusion_df)

# TODO 4: After the benchmark works, profile both functions.
# profile_table(my_fusion_fn, x, y)
# profile_table(compiled_fusion_fn, x, y)

In [ ]:
required = {"mode", "steady_ms"}
if (
    compiled_fusion_fn is not None
    and isinstance(fusion_df, pd.DataFrame)
    and len(fusion_df) == 2
    and required.issubset(fusion_df.columns)
):
    print("PASS")
else:
    print("CHECK AGAIN")

### Explain Experiment 5

1. Did your function speed up after compilation?
2. What changed in the profiler table?
3. What evidence would support the claim that fusion occurred?
4. If compilation did not help, give two plausible reasons without changing your measured result.


> **Your response**
>
> Write your response here


# Experiment 6 - Can you break graph capture?

**Concept recap:** TorchDynamo tries to capture PyTorch operations into FX graphs. If execution reaches something that cannot remain in the same captured graph, a graph break splits the region. The program can still run, but the compiler loses optimization opportunities across the boundary.

### Your Prediction
What do you expect to happen to the number of captured graphs if you deliberately insert a graph break between two groups of tensor operations?


> **Your response**
>
> Write your response here


In [ ]:
def clean_fn(a):
    y = torch.sin(a) + 1.0
    return torch.cos(y) * 2.0


# TODO: Write broken_fn so it performs similar tensor work but contains
# one explicit torch._dynamo.graph_break() between two groups of operations.
def broken_fn(a):
    return None

<details>
<summary>Hint 1</summary>

Compute one tensor intermediate, call `torch._dynamo.graph_break()`, and then continue with more tensor operations. The graph break is a statement, not a tensor operation.

</details>


<details>
<summary>Hint 2</summary>

Pseudocode:

    compute first intermediate
    explicit graph break
    compute second intermediate
    return result

</details>


In [ ]:
try:
    z = torch.randn(1024, device="cuda")
    clean_explain = dynamo.explain(clean_fn)(z)
    broken_explain = dynamo.explain(broken_fn)(z)
    clean_breaks = getattr(clean_explain, "graph_break_count", None)
    broken_breaks = getattr(broken_explain, "graph_break_count", None)
    print("Clean graph count", getattr(clean_explain, "graph_count", "unknown"))
    print("Broken graph count", getattr(broken_explain, "graph_count", "unknown"))
    print("Clean graph breaks", clean_breaks)
    print("Broken graph breaks", broken_breaks)
    if broken_breaks is not None and broken_breaks >= 1:
        print("PASS")
    else:
        print("CHECK AGAIN")
except Exception as error:
    print("CHECK AGAIN")
    print(type(error).__name__, str(error)[:500])

### Task 6B - Test fullgraph mode

Compile `broken_fn` with `fullgraph=True` and call it once inside a try/except block. Record what happens and explain why.


In [ ]:
# TODO: Compile broken_fn with fullgraph=True and call it once.
# Catch the exception so the notebook can continue.

<details>
<summary>Hint 1</summary>

`fullgraph=True` asks torch.compile to capture the function as one graph. Your function deliberately contains a boundary that prevents that.

</details>


### Explain Experiment 6

1. How did the graph count or graph break count change?
2. Why can graph breaks reduce optimization opportunities even when the Python program remains correct?
3. What did `fullgraph=True` reveal?


> **Your response**
>
> Write your response here


# torch.compile() synthesis

Complete the table from both notebooks.

| Lecture concept | Practical evidence you collected | What the evidence means |
|---|---|---|
| Eager execution | | |
| JIT compilation cost | | |
| Operator fusion | | |
| Graph break | | |


## Extra practice if you finish early

The lecture also discusses guards. A compiled graph is valid only while its assumptions hold. One possible assumption is an input shape. If you finish early, use the provided counting backend to investigate what happens when shapes change under `dynamic=False`.


In [ ]:
compile_records = []


def counting_backend(gm, example_inputs):
    compile_records.append(1)
    return gm.forward


def shape_fn(a):
    return torch.sin(a) * 2.0 + 1.0


# TODO optional:
# 1. torch._dynamo.reset()
# 2. compile shape_fn with backend=counting_backend and dynamic=False
# 3. call it with shapes 128x128, 128x128, 256x256, 256x256, 512x512
# 4. print len(compile_records) after each call
# 5. explain why repeated shapes behave differently from new shapes